In [1]:
!git clone https://github.com/dshipley71/mcp-rag.git

Cloning into 'mcp-rag'...
remote: Enumerating objects: 285, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 285 (delta 51), reused 0 (delta 0), pack-reused 184 (from 1)
Receiving objects: 100% (285/285), 129.05 KiB | 4.45 MiB/s, done.
Resolving deltas: 100% (141/141), done.


In [2]:
%cd mcp-rag

/content/mcp-rag


In [3]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 339.2/339.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 725.0/725.0 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 705.5/705.5 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.3/123.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331

In [4]:
!apt-get remove -y nodejs npm
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs
!node -v

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'npm' is not installed, so not removed
Package 'nodejs' is not installed, so not removed
0 upgraded, 0 newly installed, 0 to remove and 7 not upgraded.
2026-04-02 18:53:38 - Installing pre-requisites
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/un

In [5]:
from pathlib import Path
REPO_ROOT = Path('/content/mcp-rag').resolve()
DOCS_DIR = REPO_ROOT / 'docs'
DB_DIR = REPO_ROOT / '.rag' / 'velocirag'
print(REPO_ROOT, DOCS_DIR, DB_DIR)

/content/mcp-rag /content/mcp-rag/docs /content/mcp-rag/.rag/velocirag


In [6]:
import shutil, subprocess
if DB_DIR.exists(): shutil.rmtree(DB_DIR)
result = subprocess.run(['velocirag','index',str(DOCS_DIR)], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

Found 1 markdown files
Indexing /content/mcp-rag/docs...

Results:
  Files processed: 1
  Files skipped: 0 (unchanged)
  Chunks added: 19
  Total documents: 19
  Processing time: 7.6s

Building knowledge graph...
Graph build complete:
  Nodes: 2
  Edges: 1
  Time: 0.3s

Metadata extraction:
  Documents: 1
  Tags extracted: 0
  Cross-refs: 0

Index ready for search.




In [7]:
import nest_asyncio
nest_asyncio.apply()

In [8]:
from src.mcp_runtime import MCPRuntime
runtime = MCPRuntime(db_dir=str(DB_DIR), docs_dir=str(DOCS_DIR))
await runtime.connect()
print(await runtime.velocirag.list_tools())
print(await runtime.filesystem.list_tools())

['search', 'index', 'add_document', 'health', 'list_sources']
['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories']


In [9]:
import rich
health = await runtime.velocirag.call_tool('health', {})
rich.print(health)

CallToolResult(
    meta=None,
    content=[
        TextContent(
            type='text',
            text='{"total_documents":0,"total_chunks":0,"index_dimensions":null,"graph_nodes":0,"graph_edges":0,"la
yers":[],"model_name":"all-MiniLM-L6-v2","db_path":"/content/mcp-rag/.rag/velocirag","components":{"vector_store":t
rue,"graph_store":false,"metadata_store":false,"unified_search":true}}',
            annotations=None,
            meta=None
        )
    ],
    structuredContent={
        'total_documents': 0,
        'total_chunks': 0,
        'index_dimensions': None,
        'graph_nodes': 0,
        'graph_edges': 0,
        'layers': [],
        'model_name': 'all-MiniLM-L6-v2',
        'db_path': '/content/mcp-rag/.rag/velocirag',
        'components': {
            'vector_store': True,
            'graph_store': False,
            'metadata_store': False,
            'unified_search': True
        }
    },
    isError=False
)

In [10]:
search = await runtime.velocirag.call_tool('search', {'query':'VelociRAG','limit':3})
rich.print(search)

CallToolResult(
    meta=None,
    content=[
        TextContent(
            type='text',
            text='{"error":"No documents indexed. Use the index tool to add documents 
first.","results":[],"total_results":0,"search_time_ms":0}',
            annotations=None,
            meta=None
        )
    ],
    structuredContent={
        'error': 'No documents indexed. Use the index tool to add documents first.',
        'results': [],
        'total_results': 0,
        'search_time_ms': 0
    },
    isError=False
)

In [ ]:
result = await runtime.filesystem.call_tool('read_text_file', {'path': str(DOCS_DIR / 'VelociRAG.md')})
rich.print(result)

In [ ]:
hits = await runtime.velocirag.call_tool("search", {"query": "What is VelociRAG?", "limit": 3})
print("RAW SEARCH:", hits)

In [ ]:
async def ingest_file(runtime, file_path):
    result = await runtime.filesystem.call_tool('read_text_file', {'path': file_path})
    text = '\n'.join([b.text for b in result.content if hasattr(b,'text')])
    await runtime.velocirag.call_tool('add_document', {'text': text, 'metadata': {'path': file_path}})

await ingest_file(runtime, str(DOCS_DIR / 'VelociRAG.md'))

In [ ]:
from src.orchestrator import run_query
from src.models import QueryRequest
result = await run_query(QueryRequest(query='What is VelociRAG?'), runtime)
print(result)

In [ ]:
try:
    await runtime.close()
except BaseException:
    pass